## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## c++

#include <bits/stdc++.h>
using namespace std;

const int MAXN = 1050;

int n, A, B;
int lowbitLen;
int p[MAXN], posi[MAXN];
vector<int> ans;

void build_pos()
{
    for(int i = 0; i < n; i++)
    {
        posi[p[i]] = i;
    }
}

void do_swap()
{
    ans.push_back(0);

    for(int i = 0; i < n; i++)
    {
        if(p[i] == A)
        {
            p[i] = B;
        }
        else if(p[i] == B)
        {
            p[i] = A;
        }
    }

    build_pos();
}

void do_add(int x)
{
    x %= n;

    if(x < 0)
    {
        x += n;
    }

    if(x == 0)
    {
        return;
    }

    ans.push_back(x);

    for(int i = 0; i < n; i++)
    {
        p[i] = (p[i] + x) % n;
    }

    build_pos();
}

void do_xor(int x)
{
    if(x == 0)
    {
        return;
    }

    ans.push_back(-x);

    for(int i = 0; i < n; i++)
    {
        p[i] = p[i] ^ x;
    }

    build_pos();
}

void get_pos(int x, int y, int &px, int &py)
{
    int d = (y - x + n - lowbitLen + n) % n;

    px = 0;
    py = 0;

    for(int step = n / 2; step >= 2 * lowbitLen; step >>= 1)
    {
        if(d >= step)
        {
            d -= step;
            py += step / 2;
        }
        else
        {
            px += step / 2;
        }
    }

    px += n / 2;
    px += (x & (lowbitLen - 1));
    py += (x & (lowbitLen - 1));
}

void change_two(int x, int y)
{
    if((x / lowbitLen) % 2 == (y / lowbitLen) % 2)
    {
        int mid;

        if((x / lowbitLen) % 2 == 0)
        {
            mid = (x & (lowbitLen - 1)) + lowbitLen;
        }
        else
        {
            mid = (x & (lowbitLen - 1));
        }

        change_two(x, mid);
        change_two(y, mid);
        change_two(x, mid);
        return;
    }

    int pA, pB, pX, pY;

    get_pos(A, B, pA, pB);
    get_pos(x, y, pX, pY);

    do_add((pX - x + n) % n);
    do_xor(pX ^ pA);
    do_add((A - pA + n) % n);

    do_swap();

    do_add((pA - A + n) % n);
    do_xor(pX ^ pA);
    do_add((x - pX + n) % n);
}

struct Node
{
    int len;
    int val[MAXN];
    vector<int> op;
};

bool make_op(Node &a)
{
    vector<int> vis(a.len + 5, 0);

    for(int i = 0; i < a.len; i++)
    {
        if(a.val[i] < 0 || a.val[i] >= a.len)
        {
            return false;
        }

        vis[a.val[i]] = 1;
    }

    for(int i = 0; i < a.len; i++)
    {
        if(vis[i] == 0)
        {
            return false;
        }
    }

    if(a.len == 1)
    {
        return true;
    }

    Node l, r;

    l.len = a.len / 2;
    r.len = a.len / 2;

    for(int i = 0; i < a.len / 2; i++)
    {
        l.val[i] = a.val[i * 2] / 2;
        r.val[i] = a.val[i * 2 + 1] / 2;
    }

    if(make_op(l) == false)
    {
        return false;
    }

    if(make_op(r) == false)
    {
        return false;
    }

    if(a.val[0] & 1)
    {
        if(a.len == 2)
        {
            a.op.push_back(1);
        }
        else
        {
            a.op.push_back(-1);
        }
    }

    int now1 = 0;

    for(int i = 0; i < (int)l.op.size(); i++)
    {
        int x = l.op[i];

        if(x > 0)
        {
            a.op.push_back(-1);
            a.op.push_back(1);
        }
        else
        {
            a.op.push_back(x * 2);
            now1 ^= (-x) * 2;
        }
    }

    if(now1 != 0)
    {
        a.op.push_back(-now1);
    }

    int now2 = 0;

    for(int i = 0; i < (int)r.op.size(); i++)
    {
        int x = r.op[i];

        if(x > 0)
        {
            a.op.push_back(1);
            a.op.push_back(-1);
        }
        else
        {
            a.op.push_back(x * 2);
            now2 ^= (-x) * 2;
        }
    }

    if((now2 & (a.len / 2)) != (now1 & (a.len / 2)))
    {
        return false;
    }

    if(now1 >= a.len / 2)
    {
        now1 -= a.len / 2;
    }

    if(now2 >= a.len / 2)
    {
        now2 -= a.len / 2;
    }

    if(now1 != now2)
    {
        return false;
    }

    vector<int> b;

    for(int i = 0; i < (int)a.op.size(); i++)
    {
        int x = a.op[i];

        if(b.empty())
        {
            b.push_back(x);
        }
        else
        {
            int last = b.back();

            if(x < 0 && last < 0)
            {
                b.pop_back();

                int v = (-last) ^ (-x);

                if(v != 0)
                {
                    b.push_back(-v);
                }
            }
            else
            {
                b.push_back(x);
            }
        }
    }

    a.op = b;

    return true;
}

int main()
{
    cin >> n;

    if(n == 0)
    {
        return 0;
    }

    cin >> A >> B;

    for(int i = 0; i < n; i++)
    {
        cin >> p[i];
    }

    build_pos();

    lowbitLen = (A - B + n) % n;
    lowbitLen = lowbitLen & (-lowbitLen);

    if(lowbitLen == 0)
    {
        lowbitLen = n;
    }

    if(lowbitLen > 1)
    {
        Node base;
        base.len = lowbitLen;

        for(int i = 0; i < n; i++)
        {
            base.val[i] = p[i] & (lowbitLen - 1);
        }

        if(make_op(base) == false)
        {
            cout << -1 << "\n";
            return 0;
        }

        for(int i = 0; i < (int)base.op.size(); i++)
        {
            int x = base.op[i];

            if(x > 0)
            {
                do_add(x);
            }
            else
            {
                do_xor(-x);
            }
        }
    }

    for(int rem = 0; rem < lowbitLen; rem++)
    {
        vector<int> v;

        for(int j = rem; j < n; j += lowbitLen)
        {
            v.push_back(p[j]);
        }

        sort(v.begin(), v.end());

        int id = 0;
        int ok = 1;

        for(int j = rem; j < n; j += lowbitLen)
        {
            if(v[id] != j)
            {
                ok = 0;
                break;
            }

            id++;
        }

        if(ok == 0)
        {
            cout << -1 << "\n";
            return 0;
        }

        for(int j = rem; j < n; j += lowbitLen)
        {
            if(p[j] != j)
            {
                change_two(j, p[j]);
            }
        }
    }

    cout << ans.size() << "\n";

    for(int i = 0; i < (int)ans.size(); i++)
    {
        int x = ans[i];

        if(x == 0)
        {
            cout << "0\n";
        }
        else if(x < 0)
        {
            cout << "1 " << -x << "\n";
        }
        else
        {
            cout << "2 " << x << "\n";
        }
    }

    return 0;
}

## B 长跑

In [1]:
#python

import sys
import heapq

data = list(map(int, sys.stdin.read().split()))
idx = 0
ans = []

while idx < len(data):
    n = data[idx]
    l = data[idx + 1]
    ma = data[idx + 2]
    money = data[idx + 3]
    idx += 4

    a = []

    for i in range(n):
        p = data[idx]
        c = data[idx + 1]
        idx += 2

        if p <= l:
            a.append([p, c])

    a.append([l, 0])
    a.sort()

    heap = []
    heapq.heappush(heap, [0, 0])

    ok = 0

    for p, c in a:
        while heap and p - heap[0][1] > ma:
            heapq.heappop(heap)

        if not heap:
            continue

        cost = heap[0][0] + c

        if p == l and cost <= money:
            ok = 1
            break

        if cost <= money:
            heapq.heappush(heap, [cost, p])

    if ok == 1:
        ans.append("Yes")
    else:
        ans.append("No")

print("\n".join(ans))

## C 最长回文

In [ ]:
# c++

#include <iostream>
#include <string>
#include <vector>
#include <algorithm>

using namespace std;

int pa[600005];
int pb[600005];

string process(string s) {
    string t = "$#";
    for (int i = 0; i < s.length(); i++) {
        t += s[i];
        t += '#';
    }
    return t;
}

void get_manacher(string t, int* p) {
    int mx = 0;
    int id = 0;
    int len = t.length();
    for (int i = 1; i < len; i++) {
        if (mx > i) {
            p[i] = min(p[2 * id - i], mx - i);
        } else {
            p[i] = 1;
        }
        while (t[i + p[i]] == t[i - p[i]]) {
            p[i]++;
        }
        if (i + p[i] > mx) {
            mx = i + p[i];
            id = i;
        }
    }
}

int main() {
    int n;
    string a, b;
    cin >> n >> a >> b;

    string sa = process(a);
    string sb = process(b);

    get_manacher(sa, pa);
    get_manacher(sb, pb);

    int max_len = 1;
    int len_a = sa.length();
    int len_b = sb.length();
    int total_len = n * 2 + 2;

    for (int i = 2; i < total_len; i++) {
        int len = max(pa[i], pb[i - 2]);

        while (i - len >= 0 && i - 2 + len < len_b && sa[i - len] == sb[i - 2 + len]) {
            len++;
        }

        if (len - 1 > max_len) {
            max_len = len - 1;
        }
    }

    cout << max_len << endl;

    return 0;
}

## D 优惠券

In [ ]:
#python

import sys

class BIT:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 2)

    def add(self, x, v):
        while x <= self.n:
            self.t[x] += v
            x += x & -x

    def ask(self, x):
        s = 0
        while x > 0:
            s += self.t[x]
            x -= x & -x
        return s

    def kth(self, k):
        pos = 0
        step = 1

        while step * 2 <= self.n:
            step *= 2

        while step > 0:
            nxt = pos + step

            if nxt <= self.n and self.t[nxt] < k:
                pos = nxt
                k -= self.t[nxt]

            step //= 2

        return pos + 1

    def lower_bound(self, x):
        if x < 1:
            x = 1

        a = self.ask(x - 1)
        b = self.ask(self.n)

        if a == b:
            return -1

        return self.kth(a + 1)


data = sys.stdin.read().split()
idx = 0
out = []

while idx < len(data):
    n = int(data[idx])
    idx += 1

    bit = BIT(n)
    last_pos = {}
    last_op = {}

    ans = -1
    flag = 1

    for i in range(1, n + 1):
        s = data[idx]
        idx += 1

        op = s[0]
        x = 0

        if op == "？":
            op = "?"

        if op != "?":
            if len(s) > 1 and s[1:].isdigit():
                x = int(s[1:])
            else:
                x = int(data[idx])
                idx += 1

        if flag == 0:
            continue

        if op == "?":
            bit.add(i, 1)
        else:
            if x in last_pos:
                if last_op[x] == op:
                    p = bit.lower_bound(last_pos[x])

                    if p == -1:
                        ans = i
                        flag = 0
                    else:
                        bit.add(p, -1)

                last_pos[x] = i
                last_op[x] = op

            else:
                if op == "O":
                    p = bit.lower_bound(0)

                    if p == -1:
                        ans = i
                        flag = 0
                    else:
                        bit.add(p, -1)

                last_pos[x] = i
                last_op[x] = op

    out.append(str(ans))

print("\n".join(out))

## E 任意点

In [ ]:
#python

n = int(input())

x = [0] * 1005
y = [0] * 1005

ans = 0

for i in range(n):
    x1, y1 = map(int, input().split())

    if x[x1] == 0 and y[y1] == 0:
        ans += 1

    if x[x1] > 0 and y[y1] > 0:
        ans -= 1

    x[x1] += 1
    y[y1] += 1

print(ans - 1)

## F 通配符匹配

In [ ]:
# c++

#include <bits/stdc++.h>
using namespace std;

typedef unsigned long long ull;

ull base_num = 131;

vector<ull> get_hash(string s)
{
    int n = s.size();
    vector<ull> h(n + 1, 0);

    for(int i = 0; i < n; i++)
    {
        h[i + 1] = h[i] * base_num + (ull)(s[i] + 1);
    }

    return h;
}

ull ask_hash(vector<ull> &h, vector<ull> &pw, int l, int len)
{
    return h[l + len] - h[l] * pw[len];
}

int main()
{
    ios::sync_with_stdio(false);
    cin.tie(0);

    string p;
    cin >> p;

    int n;
    cin >> n;

    vector<string> a(n);
    int max_len = p.size() + 5;

    for(int i = 0; i < n; i++)
    {
        cin >> a[i];
        max_len = max(max_len, (int)a[i].size() + 5);
    }

    vector<ull> pw(max_len + 5);
    pw[0] = 1;

    for(int i = 1; i < (int)pw.size(); i++)
    {
        pw[i] = pw[i - 1] * base_num;
    }

    p += char(1);

    string pp = p;
    pp[pp.size() - 1] = '?';

    vector<int> pos;
    pos.push_back(-1);

    for(int i = 0; i < (int)pp.size(); i++)
    {
        if(pp[i] == '*' || pp[i] == '?')
        {
            pos.push_back(i);
        }
    }

    vector<ull> hp = get_hash(pp);

    for(int ii = 0; ii < n; ii++)
    {
        string s = a[ii];
        s += char(1);

        int m = s.size();

        vector<ull> hs = get_hash(s);

        vector<char> now(m + 1, 0);
        vector<char> nxt(m + 1, 0);

        now[0] = 1;

        for(int i = 0; i + 1 < (int)pos.size(); i++)
        {
            if(pos[i] != -1 && pp[pos[i]] == '*')
            {
                for(int j = 1; j <= m; j++)
                {
                    if(now[j - 1])
                    {
                        now[j] = 1;
                    }
                }
            }

            for(int j = 0; j <= m; j++)
            {
                nxt[j] = 0;
            }

            int l = pos[i] + 1;
            int r = pos[i + 1] - 1;
            int len = r - l + 1;

            ull need = 0;

            if(len > 0)
            {
                need = ask_hash(hp, pw, l, len);
            }

            for(int j = 0; j + len <= m; j++)
            {
                if(now[j] == 0)
                {
                    continue;
                }

                int ok = 0;

                if(len == 0)
                {
                    ok = 1;
                }
                else
                {
                    ull temp = ask_hash(hs, pw, j, len);
                    if(temp == need)
                    {
                        ok = 1;
                    }
                }

                if(ok)
                {
                    int to = j + len;

                    if(pp[pos[i + 1]] == '?')
                    {
                        to++;
                    }

                    if(to <= m)
                    {
                        nxt[to] = 1;
                    }
                }
            }

            now = nxt;
        }

        if(now[m])
        {
            cout << "YES\n";
        }
        else
        {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#python

import sys

def check(num, order):
    p = [0] * (num + 1)
    last = 0
    step = 0

    while True:
        ok1 = 1
        ok2 = 1

        for i in range(1, num + 1):
            if p[i] != 1:
                ok1 = 0
            if p[i] != 2:
                ok2 = 0

        if ok1 == 1 or ok2 == 1:
            return step

        top = [-1, -1, -1]

        for i in range(num, 0, -1):
            top[p[i]] = i

        for op in order:
            x = ord(op[0]) - ord('A')
            y = ord(op[1]) - ord('A')

            if top[x] == -1:
                continue

            d = top[x]

            if d == last:
                continue

            if top[y] == -1 or d < top[y]:
                p[d] = y
                last = d
                step += 1
                break


data = sys.stdin.read().split()

n = int(data[0])
order = data[1:]

x = check(2, order)
y = check(3, order)

if x == 5:
    ans = 2 * (3 ** (n - 1)) - 1
elif y == 7:
    ans = 2 ** n - 1
else:
    ans = 3 ** (n - 1)

print(ans)

## H 马步距离

In [ ]:
#python

from collections import deque

g = [(1, 2), (1, -2), (-1, 2), (-1, -2), (2, 1), (2, -1), (-2, 1), (-2, -1)]

x, y, a, b = map(int, input().split())

x = abs(x - a)
y = abs(y - b)

s = 0

while x + y >= 50:
    if x < y:
        x, y = y, x

    x -= 4

    if x - 4 <= y * 2:
        y -= 2

    s += 2

x += 50
y += 50

vis = [[0] * 105 for _ in range(105)]
d = [[0] * 105 for _ in range(105)]

q = deque()
q.append((x, y))
vis[x][y] = 1

while vis[50][50] == 0:
    now = q.popleft()
    xx = now[0]
    yy = now[1]

    for i in range(8):
        nx = xx + g[i][0]
        ny = yy + g[i][1]

        if nx >= 0 and nx <= 100 and ny >= 0 and ny <= 100 and vis[nx][ny] == 0:
            vis[nx][ny] = 1
            d[nx][ny] = d[xx][yy] + 1
            q.append((nx, ny))

print(s + d[50][50])

## I 直方图最大矩形

In [ ]:
#python

from typing import List

def walk2(heights):
    if len(heights) == 0:
        return 0

    heights.append(0)
    stack = []
    ans = 0

    for i in range(len(heights)):
        while len(stack) > 0 and heights[i] < heights[stack[-1]]:
            h = heights[stack.pop()]

            if len(stack) == 0:
                w = i
            else:
                w = i - stack[-1] - 1

            ans = max(ans, h * w)

        stack.append(i)

    heights.pop()
    return ans


class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        maxSize = walk2(heights)
        return maxSize

## J 消防局的设立

In [ ]:
#python

import sys

sys.setrecursionlimit(1000000)

INF = 2000000000

data = list(map(int, sys.stdin.read().split()))

n = data[0]

son = [[] for _ in range(n + 1)]

idx = 1
for i in range(2, n + 1):
    f = data[idx]
    idx += 1
    son[f].append(i)

F = [[0] * 5 for _ in range(n + 1)]

def dfs(now):
    F[now][0] = 1
    F[now][3] = 0
    F[now][4] = 0

    for s in son[now]:
        dfs(s)
        F[now][0] += F[s][4]
        F[now][3] += F[s][2]
        F[now][4] += F[s][3]

    if len(son[now]) == 0:
        F[now][1] = 1
        F[now][2] = 1
    else:
        F[now][1] = INF
        F[now][2] = INF

        for s in son[now]:
            f1 = F[s][0]
            f2 = F[s][1]

            for t in son[now]:
                if s == t:
                    continue
                f1 += F[t][3]
                f2 += F[t][2]

            F[now][1] = min(F[now][1], f1)
            F[now][2] = min(F[now][2], f2)

    for i in range(1, 5):
        F[now][i] = min(F[now][i], F[now][i - 1])

dfs(1)

print(F[1][2])